In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import scipy.sparse as sp
import os
from sklearn.datasets import load_svmlight_file
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif

2026-03-17 14:50:15.516102: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
dataset_path = '/home/ehab/Downloads/gassensordataset/Dataset/'
batch_files = [f'{dataset_path}batch{i}.dat' for i in range(1, 11)]

X_parts, y_parts = [], []
for f in batch_files:
    X_b, y_b = load_svmlight_file(f, n_features=128)
    X_parts.append(X_b)
    y_parts.append(y_b)

X = sp.vstack(X_parts).toarray()
y = np.concatenate(y_parts)

data = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(1, 129)])
data.insert(0, 'label', y.astype(int))

print(f"Combined dataset shape: {data.shape}")
print(f"Class distribution:\n{data['label'].value_counts().sort_index()}")
data.head()

Combined dataset shape: (13910, 129)
Class distribution:
label
1    2565
2    2926
3    1641
4    1936
5    3009
6    1833
Name: count, dtype: int64


,label,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_119,feature_120,feature_121,feature_122,feature_123,feature_124,feature_125,feature_126,feature_127,feature_128
0,1,15596.1621,1.868245,2.371604,2.803678,7.512213,-2.739388,-3.344671,-4.847512,15326.6914,...,-1.071137,-3.037772,3037.0390,3.972203,0.527291,0.728443,1.445783,-0.545079,-0.902241,-2.654529
1,1,26402.0704,2.532401,5.411209,6.509906,7.658469,-4.722217,-5.817651,-7.518333,23855.7812,...,-1.530519,-1.994993,4176.4453,4.281373,0.980205,1.628050,1.951172,-0.889333,-1.323505,-1.749225
2,1,42103.5820,3.454189,8.198175,10.508439,11.611003,-7.668313,-9.478675,-12.230939,37562.3008,...,-2.384784,-2.867291,5914.6685,5.396827,1.403973,2.476956,3.039841,-1.334558,-1.993659,-2.348370
3,1,42825.9883,3.451192,12.113940,16.266853,39.910056,-7.849409,-9.689894,-11.921704,38379.0664,...,-2.607199,-3.058086,6147.4744,5.501071,1.981933,3.569823,4.049197,-1.432205,-2.146158,-2.488957
4,1,58151.1757,4.194839,11.455096,15.715298,17.654915,-11.083364,-13.580692,-16.407848,51975.5899,...,-3.594763,-4.181920,8158.6449,7.174334,1.993808,3.829303,4.402448,-1.930107,-2.931265,-4.088756


In [3]:
print("Number of samples  :", X.shape[0])
print("Number of features :", X.shape[1])
print("Number of classes  :", len(np.unique(y.astype(int))))
print("Class labels       :", np.unique(y.astype(int)))

Number of samples  : 13910
Number of features : 128
Number of classes  : 6
Class labels       : [1 2 3 4 5 6]


In [4]:
# Check for missing values
print("Missing values in X:", data.iloc[:, 1:].isna().sum().sum())
print("Missing values in y:", data['label'].isna().sum())

Missing values in X: 0
Missing values in y: 0


In [5]:
# Make labels start from 0
y_clean = y.astype(int) - 1

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y_clean, test_size=0.2, random_state=42, stratify=y_clean
)

print("Training samples:", X_train.shape[0])
print("Test samples    :", X_test.shape[0])

Training samples: 11128
Test samples    : 2782


In [6]:
# Normalize using Standard Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print("Scaling complete.")
print("X_train mean (should be ~0):", round(X_train.mean(), 4))
print("X_train std  (should be ~1):", round(X_train.std(), 4))

Scaling complete.
X_train mean (should be ~0): -0.0
X_train std  (should be ~1): 1.0


In [7]:
# Apply SelectKBest with ANOVA F-score — select top 20 features
selector = SelectKBest(score_func=f_classif, k=20)
X_train = selector.fit_transform(X_train, y_train)
X_test  = selector.transform(X_test)

print("Features before selection: 128")
print("Features after selection :", X_train.shape[1])
print("Selected feature indices :", selector.get_support(indices=True))

Features before selection: 128
Features after selection : 20
Selected feature indices : [  0   2   3   8  10  11  13  14  64  66  67  68  69  72  74  76  83  91
  99 115]


In [17]:
model = keras.Sequential([
    keras.layers.Dense(128, activation='relu', input_shape=(20,)),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(6,  activation='softmax')  # 6 gas classes
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

/home/ehab/.local/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 128)            │         2,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,334 (44.27 KB)

 Trainable params: 11,334 (44.27 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
model.fit(X_train, y_train, epochs=50, validation_data=(X_test, y_test))

Epoch 1/50
348/348 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7980 - loss: 0.6391 - val_accuracy: 0.9349 - val_loss: 0.3088
Epoch 2/50
348/348 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9357 - loss: 0.2682 - val_accuracy: 0.9231 - val_loss: 0.2211
Epoch 3/50
348/348 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9517 - loss: 0.2015 - val_accuracy: 0.9526 - val_loss: 0.1598
Epoch 4/50
348/348 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9650 - loss: 0.1574 - val_accuracy: 0.9752 - val_loss: 0.1204
Epoch 5/50
348/348 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9671 - loss: 0.1386 - val_accuracy: 0.9788 - val_loss: 0.1074
Epoch 6/50
348/348 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9721 - loss: 0.1203 - val_accuracy: 0.9792 - val_loss: 0.1095
Epoch 7/50
348/348 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9738 - loss: 0.1068 - val_accuracy: 0.9856 - val_loss: 0.0852
Epoch 8/50
348/348 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9775 - loss: 0.0926 - val_accuracy: 0.

In [19]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Loss:", round(loss, 4))
print("Test Accuracy:", round(accuracy, 4))

87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9932 - loss: 0.0570
Test Loss: 0.057
Test Accuracy: 0.9932


In [20]:
# Convert to TFLite (float32 — no quantization)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("model.tflite", "wb") as f:
    f.write(tflite_model)

print("Float32 TFLite model saved.")

INFO:tensorflow:Assets written to: /tmp/tmpfieg5r9r/assets


INFO:tensorflow:Assets written to: /tmp/tmpfieg5r9r/assets


Saved artifact at '/tmp/tmpfieg5r9r'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 20), dtype=tf.float32, name='keras_tensor_4')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  138190454909488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138185208087120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138185208094160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138185208090288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138185063410928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138185063411280: TensorSpec(shape=(), dtype=tf.resource, name=None)


W0000 00:00:1773752199.413159   46746 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1773752199.413188   46746 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.


Float32 TFLite model saved.


2026-03-17 14:56:39.413561: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpfieg5r9r
2026-03-17 14:56:39.414494: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-03-17 14:56:39.414507: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpfieg5r9r
2026-03-17 14:56:39.421673: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-03-17 14:56:39.466494: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpfieg5r9r
2026-03-17 14:56:39.481528: I tensorflow/cc/saved_model/loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 67986 microseconds.


In [21]:
# Apply default optimization (int8 quantization)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model_quant = converter.convert()

with open("model_quant.tflite", "wb") as f:
    f.write(tflite_model_quant)

print("Quantized TFLite model saved.")

INFO:tensorflow:Assets written to: /tmp/tmp6d8dsqhx/assets


INFO:tensorflow:Assets written to: /tmp/tmp6d8dsqhx/assets


Saved artifact at '/tmp/tmp6d8dsqhx'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 20), dtype=tf.float32, name='keras_tensor_4')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  138190454909488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138185208087120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138185208094160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138185208090288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138185063410928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138185063411280: TensorSpec(shape=(), dtype=tf.resource, name=None)
Quantized TFLite model saved.


W0000 00:00:1773752203.730878   46746 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1773752203.730893   46746 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-03-17 14:56:43.731031: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp6d8dsqhx
2026-03-17 14:56:43.731408: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-03-17 14:56:43.731413: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmp6d8dsqhx
2026-03-17 14:56:43.733989: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-03-17 14:56:43.748545: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmp6d8dsqhx
2026-03-17 14:56:43.753718: I tensorflow/cc/saved_model/loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 22689 microseconds.


In [ ]:
# Model size comparison
float32_size = os.path.getsize("model.tflite") / 1024
quant_size   = os.path.getsize("model_quant.tflite") / 1024

print("Float32 model size:", round(float32_size, 2), "KB")
print("Quantized model size:", round(quant_size, 2), "KB")
print("Size reduction:", round((1 - quant_size / float32_size) * 100, 1), "%")

Float32 model size : 46.36 KB
Quantized model size: 17.15 KB
Size reduction      : 63.0 %


In [23]:
# Float32 model inference
interpreter_float = tf.lite.Interpreter(model_path="model.tflite")
interpreter_float.allocate_tensors()

input_details_f  = interpreter_float.get_input_details()
output_details_f = interpreter_float.get_output_details()

sample = np.array([X_test[0]], dtype=np.float32)
interpreter_float.set_tensor(input_details_f[0]['index'], sample)
interpreter_float.invoke()
output_float = interpreter_float.get_tensor(output_details_f[0]['index'])

print("Float32 model prediction:", np.argmax(output_float))

Float32 model prediction: 1


/home/ehab/.local/lib/python3.10/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [24]:
# Quantized model inference
interpreter_quant = tf.lite.Interpreter(model_path="model_quant.tflite")
interpreter_quant.allocate_tensors()

input_details_q  = interpreter_quant.get_input_details()
output_details_q = interpreter_quant.get_output_details()

interpreter_quant.set_tensor(input_details_q[0]['index'], sample)
interpreter_quant.invoke()
output_quant = interpreter_quant.get_tensor(output_details_q[0]['index'])

print("Quantized model prediction:", np.argmax(output_quant))

Quantized model prediction: 1


In [ ]:
print("Actual label: ", y_test[0])
print("Float32 prediction: ", np.argmax(output_float))
print("Quantized prediction: ", np.argmax(output_quant))

Actual label:  1
Float32 prediction:  1
Quantized prediction:  1
